[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_07/SFM_ch7_adaptive_markets/SFM_ch7_adaptive_markets.ipynb)

# SFM_ch7_adaptive_markets

Efficiency over time (the adaptive markets hypothesis of Lo, 2004): VR(5) and the robust Z*(5) on rolling windows of 500 trading days for the S&P 500, the BET and Bitcoin; three sub-periods for each; the BET in the five years before and after its upgrade to Secondary Emerging market by FTSE Russell (September 2020).

*Statistics of Financial Markets (SFM), Chapter 7: efficient markets, random walk and variance-ratio tests. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_7'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Analysis

In [ ]:
import statsmodels.api as sm

from statsmodels.tsa.stattools import adfuller, kpss

from statsmodels.tsa.adfvalues import mackinnoncrit, mackinnonp

EXTRA_SERIES = {'bvsp': ('BVSP.INDX', 'Bovespa', 'Equity', 'close', '2000-01-01'), 'nifty': ('NSEI.INDX', 'Nifty 50', 'Equity', 'close', '2000-01-01'), 'bist': ('XU100.INDX', 'BIST 100', 'Equity', 'close', '2000-01-01'), 'ssec': ('SSEC.INDX', 'Shanghai Composite', 'Equity', 'close', '2000-01-01'), 'kospi': ('KS11.INDX', 'KOSPI', 'Equity', 'close', '2000-01-01'), 'ipc': ('MXX.INDX', 'IPC Mexico', 'Equity', 'close', '2000-01-01')}

NAME = {'sp500': 'S&P 500', 'dax': 'DAX', 'bet': 'BET', 'btc': 'Bitcoin', 'eth': 'Ethereum', 'stoxx50': 'Euro Stoxx 50', 'nikkei': 'Nikkei 225', 'tlv': 'Banca Transilvania', 'snp': 'OMV Petrom', 'brd': 'BRD', 'tgn': 'Transgaz', 'wig20': 'WIG20', 'bux': 'BUX', 'px': 'PX', 'bvsp': 'Bovespa', 'nifty': 'Nifty 50', 'bist': 'BIST 100', 'ssec': 'Shanghai Composite', 'kospi': 'KOSPI', 'ipc': 'IPC Mexico'}

START = {'sp500': '1990-01-01', 'dax': '1990-01-01', 'bet': '1997-01-01', 'btc': None, 'tlv': '2010-01-01', 'snp': '2010-01-01', 'brd': '2010-01-01', 'tgn': '2010-01-01'}

ASSETS = ['sp500', 'dax', 'bet', 'btc']

STOCKS = ['tlv', 'snp', 'brd', 'tgn']

DEVELOPED = ['sp500', 'dax', 'stoxx50', 'nikkei']

EMERGING = ['bet', 'wig20', 'bux', 'px', 'bist', 'bvsp', 'ipc', 'nifty', 'kospi', 'ssec']

CRYPTO = ['btc', 'eth']

MARKETS_START = '2016-09-19'

BAD_DAYS = {'tlv': ['2016-05-30', '2016-05-31']}

COLORS = {'sp500': '#1A3A6E', 'dax': '#17A2B8', 'bet': '#CD0000', 'btc': '#B5853F', 'snp': '#2E7D32', 'brd': '#8E44AD', 'tlv': '#E67E22', 'tgn': '#DC3545'}

QS = (2, 5, 10, 20)

LB_LAGS = 10

WINDOW = 500

STEP = 21

SEED = 2026

B_BOOT = 500

FTSE_UPGRADE = '2020-09-21'

SUBPERIODS = {'bet': [('1997-01-01', '2006-12-31'), ('2007-01-01', '2016-12-31'), ('2017-01-01', None)], 'sp500': [('1990-01-01', '2001-12-31'), ('2002-01-01', '2013-12-31'), ('2014-01-01', None)], 'btc': [('2014-01-01', '2017-12-31'), ('2018-01-01', '2021-12-31'), ('2022-01-01', None)]}

DAYS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

ACF_CASES = [('AR(1), alpha = 0.9', (0.9,), ()), ('AR(1), alpha = -0.9', (-0.9,), ()), ('AR(2), alpha1 = 0.5, alpha2 = 0.4', (0.5, 0.4), ()), ('MA(1), beta = 0.5', (), (0.5,)), ('MA(1), beta = -0.5', (), (-0.5,)), ('MA(2), beta1 = 0.5, beta2 = 0.4', (), (0.5, 0.4))]

def returns(k, start=None):
    """Daily log returns in % on the series' own calendar (from `start`, else the default start of the series);
    known data errors removed."""
    if k in EXTRA_SERIES:
        SERIES.setdefault(k, EXTRA_SERIES[k])
    r = log_returns(k, start or START.get(k))
    if k in BAD_DAYS:
        r = r.drop(pd.to_datetime(BAD_DAYS[k]), errors='ignore')
    return r


def log_price(k, start=None):
    """Natural logarithm of the closing price (adjusted close for stocks)."""
    if k in EXTRA_SERIES:
        SERIES.setdefault(k, EXTRA_SERIES[k])
    p = np.log(load_close(k, start or START.get(k)))
    if k in BAD_DAYS:            # the same two days as in returns(): the late adjustment cancels out
        p = p.drop(pd.to_datetime(BAD_DAYS[k]), errors='ignore')
    return p


def acf(x, nlags):
    """Sample autocorrelations rho_hat(1..nlags) = sum (x_t - m)(x_{t-k} - m) / sum (x_t - m)^2."""
    e = np.asarray(x, float) - np.mean(x)
    s = np.sum(e ** 2)
    return np.array([np.sum(e[k:] * e[:-k]) / s for k in range(1, nlags + 1)])


def variance_ratio(x, q):
    """Lo and MacKinlay (1988): overlapping VR(q) with their bias corrections, the homoskedastic statistic
    Z(q) = (VR - 1) / sqrt(2(2q-1)(q-1)/(3qT)) and the heteroskedasticity-robust Z*(q) = (VR - 1) / sqrt(theta/T),
    theta = sum_{j=1}^{q-1} [2(q-j)/q]^2 delta_hat(j)."""
    x = np.asarray(x, float)
    T = len(x)
    mu = x.mean()
    e = x - mu
    var_a = np.sum(e ** 2) / (T - 1)
    agg = np.convolve(x, np.ones(q), mode='valid') - q * mu          # overlapping q-day returns minus q mu
    m = q * (T - q + 1) * (1 - q / T)
    var_c = np.sum(agg ** 2) / m
    vr = var_c / var_a
    z = (vr - 1) / np.sqrt(2 * (2 * q - 1) * (q - 1) / (3 * q * T))
    den = np.sum(e ** 2) ** 2
    theta = sum((2 * (q - j) / q) ** 2 * T * np.sum(e[j:] ** 2 * e[:-j] ** 2) / den for j in range(1, q))
    zs = (vr - 1) / np.sqrt(theta / T)
    return {'q': q, 'vr': float(vr), 'z': float(z), 'zs': float(zs), 'p_z': float(2 * stats.norm.sf(abs(z))),
            'p_zs': float(2 * stats.norm.sf(abs(zs))), 'se_rob': float(np.sqrt(theta / T)),
            'se_iid': float(np.sqrt(2 * (2 * q - 1) * (q - 1) / (3 * q * T)))}


def chow_denning(x, qs=QS, alpha=0.05):
    """Chow and Denning (1993): CD = max_i |Z*(q_i)|; critical value and p-value from the studentized maximum
    modulus distribution with m = len(qs) and infinite degrees of freedom: P(CD <= c) = (2 Phi(c) - 1)^m."""
    zs = [variance_ratio(x, q)['zs'] for q in qs]
    cd = float(np.max(np.abs(zs)))
    m = len(qs)
    crit = float(stats.norm.ppf(1 - (1 - (1 - alpha) ** (1 / m)) / 2))
    return {'cd': cd, 'p': float(1 - (2 * stats.norm.cdf(cd) - 1) ** m), 'crit': crit, 'm': m}


def rolling_vr(k, q=5, window=WINDOW, step=STEP):
    """VR(q) and Z*(q) on rolling windows of `window` days, one every `step` days (dated at the window end)."""
    r = returns(k)
    x = r.values
    rows = []
    for end in range(window, len(x) + 1, step):
        v = variance_ratio(x[end - window:end], q)
        rows.append((r.index[end - 1], v['vr'], v['zs']))
    return pd.DataFrame(rows, columns=['date', 'vr', 'zs']).set_index('date')


def fig_rolling_vr(names=('sp500', 'bet', 'btc'), q=5, save=True):
    """Rolling VR(5) and Z*(5) on 500-day windows (adaptive markets: efficiency changes over time)."""
    fig, axes = plt.subplots(2, 1, figsize=(10, 5.6), sharex=True)
    out = {}
    for k in names:
        d = rolling_vr(k, q)
        axes[0].plot(d.index, d['vr'], color=COLORS[k], lw=1.3, label=NAME[k])
        axes[1].plot(d.index, d['zs'], color=COLORS[k], lw=1.1)
        out[k] = {'share_rej': float(np.mean(np.abs(d['zs']) > 1.96)), 'share_pos': float(np.mean(d['zs'] > 1.96)),
                  'share_neg': float(np.mean(d['zs'] < -1.96)), 'n_win': len(d), 'vr_min': float(d['vr'].min()),
                  'vr_max': float(d['vr'].max()), 'date_max': d['vr'].idxmax().date().isoformat(),
                  'date_min': d['vr'].idxmin().date().isoformat(), 'first': d.index[0].date().isoformat()}
    axes[0].axhline(1, color='black', lw=0.7, ls='--')
    axes[0].set_ylabel(f'VR({q})')
    axes[1].axhline(1.96, color='black', lw=0.7, ls='--')
    axes[1].axhline(-1.96, color='black', lw=0.7, ls='--')
    axes[1].set_ylabel(f'Robust Z*({q})')
    st.fig_legend_bottom(fig, handles=axes[0].get_legend_handles_labels()[0], labels=axes[0].get_legend_handles_labels()[1], ncol=3, y=0.0)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch7_rolling_vr')
    return out


def subperiods(table=SUBPERIODS, q=5):
    """rho_hat(1), VR(5), Z*(5) and the Chow-Denning p-value in three sub-periods."""
    out = {}
    for k, per in table.items():
        r = returns(k)
        out[k] = []
        for a, b in per:
            x = r.loc[a:b].values
            v = variance_ratio(x, q)
            out[k].append({'from': r.loc[a:b].index[0].date().isoformat(), 'to': r.loc[a:b].index[-1].date().isoformat(),
                           'n': len(x), 'rho1': float(acf(x, 1)[0]), 'vr': v['vr'], 'zs': v['zs'], 'cd_p': chow_denning(x)['p']})
    return out


def bet_upgrade(edge=FTSE_UPGRADE, years=5, q=5):
    """The BET in the five years before and after its upgrade to Secondary Emerging market (September 2020)."""
    r = returns('bet')
    e = pd.Timestamp(edge)
    out = {}
    for lab, a, b in [('before', e - pd.DateOffset(years=years), e - pd.Timedelta(days=1)), ('after', e, e + pd.DateOffset(years=years))]:
        x = r.loc[a:b].values
        v = variance_ratio(x, q)
        out[lab] = {'from': r.loc[a:b].index[0].date().isoformat(), 'to': r.loc[a:b].index[-1].date().isoformat(), 'n': len(x),
                    'rho1': float(acf(x, 1)[0]), 'vr': v['vr'], 'zs': v['zs'], 'se': v['se_rob'], 'cd_p': chow_denning(x)['p']}
    out['z_diff'] = (out['after']['vr'] - out['before']['vr']) / np.sqrt(out['after']['se'] ** 2 + out['before']['se'] ** 2)
    return out

## Run

In [ ]:
print(fig_rolling_vr())
print(subperiods())
print(bet_upgrade())

![sfm_ch7_rolling_vr](./sfm_ch7_rolling_vr.png)

Output: `sfm_ch7_rolling_vr.pdf`